# 02 Feature Engineering

This notebook builds the **intermediate feature layer** that sits between `01_data_audit.ipynb` and `03_business_analysis.ipynb`. In the project sequence, it should be run **after Phase 1 processed tables are refreshed** and **before the main business analysis notebook**.

The goal is to make the transformation logic explicit, auditable, and reusable. When rerun, this notebook refreshes its Phase 2 feature-layer outputs in `outputs/tables/`.

The notebook focuses on three things:
- loading the latest normalized Phase 1 tables
- building title-level features used in Phase 2
- validating and saving the engineered feature layer before business analysis


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/processed/titles.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
pd.options.display.float_format = lambda value: f'{value:,.3f}'

from src.feature_engineering import (
    build_country_view,
    build_genre_view,
    build_title_features,
    load_phase1_tables,
)
from src.utils import ensure_directory, save_dataframe

## Load Phase 1 Assets

Phase 1 leaves the project with a normalized base table plus bridge tables in `data/processed/`. This notebook should always read the latest version of those files after `01_data_audit.ipynb` has been rerun.


In [2]:
tables = load_phase1_tables(PROJECT_ROOT / 'data/processed')

phase1_inventory = pd.DataFrame(
    [
        {'table_name': name, 'row_count': len(frame), 'column_count': frame.shape[1]}
        for name, frame in tables.items()
    ]
).sort_values('table_name').reset_index(drop=True)

display(phase1_inventory)

,table_name,row_count,column_count
0,title_cast,64124,2
1,title_country,10012,2
2,title_director,6977,2
3,title_genre,19323,2
4,titles,8807,15


## Build the Title-Level Feature Layer

The key design choice in Phase 2 is to move from normalized entities to a title-level analytical layer. That gives `03_business_analysis.ipynb` a stable intermediate table with freshness, breadth, and country-scope fields already computed.


In [3]:
title_features = build_title_features(
    tables['titles'],
    tables['title_country'],
    tables['title_genre'],
)
country_view = build_country_view(title_features, tables['title_country'])
genre_view = build_genre_view(title_features, tables['title_genre'])

display(title_features[['show_id', 'title', 'type', 'release_year', 'date_added_year', 'country_count', 'genre_count', 'country_scope', 'release_to_add_lag', 'release_to_add_lag_clean']].head(8))

,show_id,title,type,release_year,date_added_year,country_count,genre_count,country_scope,release_to_add_lag,release_to_add_lag_clean
0,s1,Dick Johnson Is Dead,Movie,2020,2021,1,1,Single-country,1,1
1,s10,The Starling,Movie,2021,2021,1,2,Single-country,0,0
2,s100,On the Verge,TV Show,2021,2021,2,2,Multi-country,0,0
3,s1000,Stowaway,Movie,2021,2021,2,3,Multi-country,0,0
4,s1001,Wild Dog,Movie,2020,2021,0,2,Missing country,1,1
5,s1002,Oloibiri,Movie,2015,2021,3,3,Multi-country,6,6
6,s1003,Tell Me When,Movie,2021,2021,1,2,Single-country,0,0
7,s1004,Zero,TV Show,2021,2021,1,3,Single-country,0,0


In [4]:
feature_dictionary = pd.DataFrame(
    [
        {'feature_name': 'country_count', 'level': 'title', 'definition': 'Number of normalized country tags attached to a title', 'business_use': 'Measure supply breadth and identify single-country versus multi-country titles'},
        {'feature_name': 'genre_count', 'level': 'title', 'definition': 'Number of normalized genre tags attached to a title', 'business_use': 'Measure category breadth and support co-occurrence analysis'},
        {'feature_name': 'release_to_add_lag', 'level': 'title', 'definition': 'Raw difference between year_added and release_year', 'business_use': 'Quantify how quickly titles enter the catalog after release'},
        {'feature_name': 'is_valid_release_to_add_lag', 'level': 'title', 'definition': 'Flag for non-negative release-to-add lag', 'business_use': 'Separate usable freshness signals from metadata anomalies'},
        {'feature_name': 'release_to_add_lag_clean', 'level': 'title', 'definition': 'Release-to-add lag with negative values removed', 'business_use': 'Primary freshness metric for Phase 2 reporting'},
        {'feature_name': 'country_scope', 'level': 'title', 'definition': 'Missing country, Single-country, or Multi-country title scope', 'business_use': 'Support geographic footprint and co-production analysis'},
        {'feature_name': 'is_recent_within_1y', 'level': 'title', 'definition': 'Title added within 1 year of release', 'business_use': 'Measure near-current catalog freshness'},
        {'feature_name': 'is_recent_within_3y', 'level': 'title', 'definition': 'Title added within 3 years of release', 'business_use': 'Primary recent-share freshness KPI'},
        {'feature_name': 'is_recent_within_5y', 'level': 'title', 'definition': 'Title added within 5 years of release', 'business_use': 'Broader freshness KPI for slower-moving categories'},
    ]
)

display(feature_dictionary)

,feature_name,level,definition,business_use
0,country_count,title,Number of normalized country tags attached to ...,Measure supply breadth and identify single-cou...
1,genre_count,title,Number of normalized genre tags attached to a ...,Measure category breadth and support co-occurr...
2,release_to_add_lag,title,Raw difference between year_added and release_...,Quantify how quickly titles enter the catalog ...
3,is_valid_release_to_add_lag,title,Flag for non-negative release-to-add lag,Separate usable freshness signals from metadat...
4,release_to_add_lag_clean,title,Release-to-add lag with negative values removed,Primary freshness metric for Phase 2 reporting
5,country_scope,title,"Missing country, Single-country, or Multi-coun...",Support geographic footprint and co-production...
6,is_recent_within_1y,title,Title added within 1 year of release,Measure near-current catalog freshness
7,is_recent_within_3y,title,Title added within 3 years of release,Primary recent-share freshness KPI
8,is_recent_within_5y,title,Title added within 5 years of release,Broader freshness KPI for slower-moving catego...


## QA Checks

Before the business analysis notebook uses this layer, we verify that title coverage is preserved, anomalies are visible, and bridge views line up with the source counts. After validation, the notebook writes refreshed feature-layer tables to `outputs/tables/`.


In [5]:
feature_qa = pd.DataFrame(
    [
        {'check_name': 'titles_row_count', 'value': len(title_features)},
        {'check_name': 'distinct_show_id', 'value': title_features['show_id'].nunique()},
        {'check_name': 'negative_lag_titles', 'value': int((title_features['release_to_add_lag'].dropna() < 0).sum())},
        {'check_name': 'valid_lag_titles', 'value': int(title_features['release_to_add_lag_clean'].notna().sum())},
        {'check_name': 'missing_country_titles', 'value': int(title_features['country_count'].eq(0).sum())},
        {'check_name': 'single_country_titles', 'value': int(title_features['country_count'].eq(1).sum())},
        {'check_name': 'multi_country_titles', 'value': int(title_features['country_count'].gt(1).sum())},
        {'check_name': 'country_view_rows', 'value': len(country_view)},
        {'check_name': 'genre_view_rows', 'value': len(genre_view)},
    ]
)

feature_distribution_summary = pd.DataFrame(
    [
        {
            'metric': 'country_count',
            'mean': title_features['country_count'].mean(),
            'median': title_features['country_count'].median(),
            'p75': title_features['country_count'].quantile(0.75),
            'max': title_features['country_count'].max(),
        },
        {
            'metric': 'genre_count',
            'mean': title_features['genre_count'].mean(),
            'median': title_features['genre_count'].median(),
            'p75': title_features['genre_count'].quantile(0.75),
            'max': title_features['genre_count'].max(),
        },
        {
            'metric': 'release_to_add_lag_clean',
            'mean': title_features['release_to_add_lag_clean'].mean(),
            'median': title_features['release_to_add_lag_clean'].median(),
            'p75': title_features['release_to_add_lag_clean'].quantile(0.75),
            'max': title_features['release_to_add_lag_clean'].max(),
        },
    ]
)

country_scope_summary = (
    title_features['country_scope']
    .value_counts(dropna=False)
    .rename_axis('country_scope')
    .reset_index(name='title_count')
)
country_scope_summary['share'] = country_scope_summary['title_count'] / country_scope_summary['title_count'].sum()

display(feature_qa)
display(feature_distribution_summary)
display(country_scope_summary)

,check_name,value
0,titles_row_count,8807
1,distinct_show_id,8807
2,negative_lag_titles,14
3,valid_lag_titles,8783
4,missing_country_titles,831
5,single_country_titles,6661
6,multi_country_titles,1315
7,country_view_rows,10012
8,genre_view_rows,19323


,metric,mean,median,p75,max
0,country_count,1.137,1.000,1.000,12
1,genre_count,2.194,2.000,3.000,3
2,release_to_add_lag_clean,4.698,1.000,5.000,93


,country_scope,title_count,share
0,Single-country,6661,0.756
1,Multi-country,1315,0.149
2,Missing country,831,0.094


**Interpretation**

- The title-level feature layer preserves full title coverage and keeps anomalies explicit rather than silently dropping them.
- Negative lag records are rare, which makes `release_to_add_lag_clean` a defensible business metric for freshness analysis.
- Most titles are single-country and multi-genre, which is exactly the structure the Phase 2 catalog and geographic analyses need.

## Bridge Views for Downstream Analysis

The final step is to project the engineered title-level features back onto the normalized entity views. This is how the business analysis notebook can answer questions like freshness by country or rating mix by genre without rebuilding transformations inline.

In [6]:
display(country_view[['show_id', 'country', 'type', 'country_scope', 'release_to_add_lag_clean']].head(8))
display(genre_view[['show_id', 'genre', 'type', 'rating_group', 'release_to_add_lag_clean']].head(8))

,show_id,country,type,country_scope,release_to_add_lag_clean
0,s1,United States,Movie,Single-country,1
1,s10,United States,Movie,Single-country,0
2,s100,France,TV Show,Multi-country,0
3,s100,United States,TV Show,Multi-country,0
4,s1000,Germany,Movie,Multi-country,0
5,s1000,United States,Movie,Multi-country,0
6,s1002,Canada,Movie,Multi-country,6
7,s1002,Nigeria,Movie,Multi-country,6


,show_id,genre,type,rating_group,release_to_add_lag_clean
0,s1,Documentaries,Movie,Teen,1
1,s10,Comedies,Movie,Teen,0
2,s10,Dramas,Movie,Teen,0
3,s100,TV Comedies,TV Show,Mature,0
4,s100,TV Dramas,TV Show,Mature,0
5,s1000,Dramas,Movie,Mature,0
6,s1000,International Movies,Movie,Mature,0
7,s1000,Thrillers,Movie,Mature,0


In [7]:
TABLES_DIR = ensure_directory(PROJECT_ROOT / 'outputs/tables')
feature_output_tables = {
    'phase2_feature_dictionary.csv': feature_dictionary,
    'phase2_feature_qa_summary.csv': feature_qa,
    'phase2_feature_distribution_summary.csv': feature_distribution_summary,
}
for file_name, frame in feature_output_tables.items():
    save_dataframe(frame, TABLES_DIR / file_name)

feature_output_inventory = pd.DataFrame(
    {'file_name': sorted(feature_output_tables.keys())}
)
display(feature_output_inventory)
print(f'Loaded Phase 1 processed tables from: {PROJECT_ROOT / 'data/processed'}')
print(f'Saved {len(feature_output_tables)} Phase 2 feature tables to: {TABLES_DIR}')
print('These refreshed tables should be in place before rerunning 03_business_analysis.ipynb')


,file_name
0,phase2_feature_dictionary.csv
1,phase2_feature_distribution_summary.csv
2,phase2_feature_qa_summary.csv


Loaded Phase 1 processed tables from: /Users/xinyue/Documents/projects/netflix_da/data/processed
Saved 3 Phase 2 feature tables to: /Users/xinyue/Documents/projects/netflix_da/outputs/tables
These refreshed tables should be in place before rerunning 03_business_analysis.ipynb


## Outcome

This notebook completes the missing intermediate layer in the repo structure:

- `01_data_audit.ipynb` validates the raw and processed data foundation
- `02_feature_engineering.ipynb` builds and validates the title-level feature layer
- `03_business_analysis.ipynb` consumes that layer for business-facing analysis

That separation makes the project easier to review and closer to a professional analytics workflow.